# Extension: does the finding generalise? (2nd model + DoRA)

The original study found that on **Qwen2.5-1.5B / Alpaca**, LoRA makes *tiny (≈0.4–1.5%), high-rank, attention-biased edits that rescale rather than rotate representations* — but it was explicitly exploratory: one model, one PEFT variant.

This notebook turns that anecdote into a claim (or refutes it) along two axes:

1. **A second model family** — [SmolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B) (ungated, Llama-style architecture).
2. **A second PEFT variant** — **DoRA** (Liu et al., 2024), which decomposes the update into magnitude and direction.

It also adds the **behavioural validation** the original study lacked: held-out instruction loss (base vs fine-tuned).

**Runtime**: 4 configs (2 models × LoRA/DoRA). ~40 min on an **A100/L4**, 2–3 h on a free T4. Reduce `MAX_SAMPLES` for a faster smoke run.

> ⚠️ **Cómo ejecutarlo**: celdas en orden, de arriba abajo. Todas las celdas son *idempotentes* (puedes re-ejecutar cualquiera sin romper nada). Si algo se queda en mal estado: *Entorno de ejecución → Reiniciar* y vuelve a ejecutar desde arriba.

In [ ]:
# Setup robusto e idempotente: siempre parte de /content, clona limpio y purga
# cualquier versión vieja del paquete que el kernel tenga cacheada.
import os, shutil, sys

REPO = "/content/Interpreting-LoRA-Fine-Tuning"

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/Interpreting-LoRA-Fine-Tuning.git
os.chdir(REPO)

!pip install -q -U "transformers>=4.44" "peft>=0.13" "datasets>=2.18" accelerate

# purgar módulos viejos y dejar solo la ruta absoluta del clon nuevo
for k in list(sys.modules):
    if k.startswith("lora_interp"):
        del sys.modules[k]
sys.path[:] = [p for p in sys.path if "Interpreting-LoRA" not in str(p)]
sys.path.insert(0, f"{REPO}/src")

# verificación: si esto falla, el código clonado no es el esperado
import inspect
from lora_interp.train import train_lora
assert "use_dora" in inspect.signature(train_lora).parameters, \
    "train.py sin use_dora: haz push del repo actualizado antes de continuar"
print("setup OK — cwd:", os.getcwd())
print("train_lora:", inspect.signature(train_lora))

In [ ]:
import copy, gc, json

import numpy as np
import torch

from lora_interp.analysis import adapter_update_norms, representation_drift
from lora_interp.train import train_lora, _format
from lora_interp.utils import set_seed, load_base

MODELS = ["Qwen/Qwen2.5-1.5B", "HuggingFaceTB/SmolLM2-1.7B"]
VARIANTS = [("lora", False), ("dora", True)]
RANK = 16
TARGET = "all"
MAX_SAMPLES = 2000          # reduce a 500 para una prueba rápida
SEED = 42

HELDOUT_DRIFT = [
    "### Instruction:\nExplain what overfitting is.\n\n### Response:\n",
    "### Instruction:\nSummarise the causes of the French Revolution.\n\n### Response:\n",
    "### Instruction:\nWrite a Python function that reverses a string.\n\n### Response:\n",
    "### Instruction:\nGive three tips for sleeping better.\n\n### Response:\n",
    "### Instruction:\nWhat is the capital of Australia?\n\n### Response:\n",
]

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## Behavioural validation: held-out instruction loss

Mean token-level cross-entropy on 200 *held-out* Alpaca examples (never used in training — training takes the first `MAX_SAMPLES` of the seeded shuffle, we take the next 200).

In [ ]:
from datasets import load_dataset

_alpaca = load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=SEED)
EVAL_EXAMPLES = [_format(e) for e in _alpaca.select(range(MAX_SAMPLES, MAX_SAMPLES + 200))]


@torch.no_grad()
def heldout_loss(model, tok, texts, max_len=512):
    model.eval()
    losses = []
    for t in texts:
        ids = tok(t + tok.eos_token, return_tensors="pt", truncation=True,
                  max_length=max_len).input_ids.to(model.device)
        out = model(ids, labels=ids)
        losses.append(out.loss.item())
    return float(np.mean(losses))

print(f"{len(EVAL_EXAMPLES)} ejemplos held-out listos")

## Exact update norms (works for DoRA too)

`adapter_update_norms` computes `dW = (α/r)·B@A`, which is exact for LoRA but ignores DoRA's magnitude vector. Here we measure the *actual* update by comparing the merged fine-tuned weights against the base weights, so LoRA and DoRA are compared on equal footing.

In [ ]:
import re as _re

_TARGET_RE = _re.compile(r"(q|k|v|o|gate|up|down)_proj$")


@torch.no_grad()
def exact_update_norms(base_model, merged_model):
    base = {n: m.weight for n, m in base_model.named_modules()
            if hasattr(m, "weight") and _TARGET_RE.search(n)}
    rows = []
    for n, m in merged_model.named_modules():
        if not (hasattr(m, "weight") and _TARGET_RE.search(n)):
            continue
        W0 = base[n].float()
        dW = m.weight.float() - W0
        sv = torch.linalg.svdvals(dW)
        p = (sv / sv.sum()).clamp_min(1e-12)
        lyr = _re.search(r"\.layers\.(\d+)\.", n)
        rows.append({
            "module": n,
            "layer": int(lyr.group(1)) if lyr else -1,
            "type": n.split(".")[-1],
            "rel_update": (torch.linalg.norm(dW) / torch.linalg.norm(W0)).item(),
            "effective_rank": float(torch.exp(-(p * p.log()).sum())),
        })
    return rows

print("exact_update_norms definido")

## Run the grid (2 models × LoRA/DoRA)

Idempotente: guarda cada configuración terminada en `results_partial.json`; si la celda se interrumpe y la relanzas, salta las configuraciones ya hechas.

In [ ]:
import os

ATTN_TYPES = {"q_proj", "k_proj", "v_proj", "o_proj"}
PARTIAL = "results_partial.json"
os.makedirs("figures", exist_ok=True)

results = json.load(open(PARTIAL)) if os.path.exists(PARTIAL) else []
done = {(r["model"], r["variant"]) for r in results}
base_losses = {}

for model_name in MODELS:
    short = model_name.split("/")[-1]
    todo = [(v, d) for v, d in VARIANTS if (short, v) not in done]
    if not todo:
        print(f"{short}: ya completado, salto")
        continue

    # snapshot del modelo base para la eval conductual + dW exacto
    base_model, base_tok = load_base(model_name)
    if short not in base_losses:
        base_losses[short] = heldout_loss(base_model, base_tok, EVAL_EXAMPLES)
    base_model.cpu(); gc.collect(); torch.cuda.empty_cache()

    for variant, use_dora in todo:
        tag = f"{short}_{variant}_r{RANK}_{TARGET}"
        print(f"\n{'='*70}\n{tag}\n{'='*70}")
        model, tok = train_lora(model_name=model_name, rank=RANK, target=TARGET,
                                max_samples=MAX_SAMPLES, epochs=1.0,
                                output_dir=f"outputs/{tag}", seed=SEED,
                                use_dora=use_dora)

        # 1) drift (adapter on/off) — antes del merge
        drift = representation_drift(model, tok, HELDOUT_DRIFT)
        # 2) validación conductual
        ft_loss = heldout_loss(model, tok, EVAL_EXAMPLES)
        # 3) dW exacto vía merge
        merged = model.merge_and_unload()
        rows = exact_update_norms(base_model, merged.cpu())
        json.dump(rows, open(f"figures/exact_update_{tag}.json", "w"), indent=1)

        attn_u = [r["rel_update"] for r in rows if r["type"] in ATTN_TYPES]
        mlp_u = [r["rel_update"] for r in rows if r["type"] not in ATTN_TYPES]
        results.append({
            "model": short, "variant": variant,
            "rank": RANK, "target": TARGET,
            "heldout_loss_base": round(base_losses[short], 4),
            "heldout_loss_ft": round(ft_loss, 4),
            "mean_rel_update": round(float(np.mean([r["rel_update"] for r in rows])), 4),
            "rel_update_attn": round(float(np.mean(attn_u)), 4),
            "rel_update_mlp": round(float(np.mean(mlp_u)), 4),
            "mean_eff_rank": round(float(np.mean([r["effective_rank"] for r in rows])), 2),
            "drift_cos_final": round(drift["cosine_similarity"][-1], 4),
            "drift_relL2_final": round(drift["relative_l2"][-1], 4),
        })
        json.dump(results, open(PARTIAL, "w"), indent=1)   # checkpoint
        print(results[-1])

        del model, merged
        gc.collect(); torch.cuda.empty_cache()

    del base_model
    gc.collect(); torch.cuda.empty_cache()

print(f"\n{len(results)}/4 configuraciones completadas")

## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("results_extension.csv", index=False)
print(df.to_markdown(index=False))

## What to check against the original four findings

1. **Tiny updates** — is `mean_rel_update` still ≲ 1–2% on SmolLM2? Does DoRA's exact update stay in the same range, or does the magnitude component make it larger?
2. **Full rank budget** — is `mean_eff_rank` still ≈ r (16)? DoRA's directional update is also rank-r, but the magnitude rescaling can lift the effective rank of the *exact* dW; that in itself would be a finding.
3. **Attention bias** — is `rel_update_attn > rel_update_mlp` on both models and both variants?
4. **Rescale, not rotate** — is `drift_cos_final` still high (≥0.95) with sizeable `drift_relL2_final`? If DoRA shows *lower* cosine at matched loss, the magnitude/direction decomposition matters behaviourally.

Plus the behavioural sanity check: `heldout_loss_ft < heldout_loss_base` for every row — otherwise the run didn't adapt and its diagnostics shouldn't be interpreted.

**Next step**: paste `results_extension.csv` as a second table in the README under "Results", and update the *Limitations* paragraph — with a second model family and a second PEFT variant the findings are no longer single-model anecdotes.

> 💾 Antes de cerrar Colab, descarga `results_extension.csv` y `results_partial.json` (Archivos, panel izquierdo) — el runtime se borra al desconectar.